# Context Groundedness Scorer — Sanity Check

**Goal:** Verify that `ContextGroundednessScorer` works end-to-end with a locally-served
LLM (`qwen3:8b` via Ollama) on examples from the RAGTruth dataset.

### Pipeline overview
1. Load `qwen3:8b` as a `ChatOllama` LangChain object.
2. Load RAGTruth dataset, pick one clean and one hallucinated example.
3. Run `ContextGroundednessScorer.score()` — decomposes answer into claims,
   verifies each claim against context via NLI entailment.
4. Inspect results: claims, labels, scores.

## 0. Prerequisites

Make sure ollama is running and the model is pulled:
```bash
ollama pull qwen3:8b
```

In [2]:
import os
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11450"  # adjust to your ollama port

## 1. Load the local LLM via ChatOllama

In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.0,
)

# Quick smoke test
response = llm.invoke("Say hello in one word.")
print("LLM response:", response.content)

LLM response: Hello.


## 2. Load RAGTruth examples

In [4]:
from datasets import load_dataset
import pandas as pd
import json
import ast

ds = load_dataset("wandb/RAGTruth-processed")
df = ds["train"].to_pandas().copy()
df["answer"] = df["output"]

# Parse hallucination labels
def get_processed_counts(x):
    if isinstance(x, dict):
        return int(x.get("evident_conflict", 0) or 0), int(x.get("baseless_info", 0) or 0)
    return 0, 0

tmp = df["hallucination_labels_processed"].apply(get_processed_counts)
df["evident_conflict"] = tmp.apply(lambda t: t[0])
df["baseless_info"] = tmp.apply(lambda t: t[1])
df["has_hallucination"] = (df["evident_conflict"] > 0) | (df["baseless_info"] > 0)

print(f"Total examples: {len(df)}")
print(f"With hallucination: {df['has_hallucination'].sum()}")
print(f"Clean: {(~df['has_hallucination']).sum()}")

Repo card metadata block was not found. Setting CardData to empty.


Total examples: 15090
With hallucination: 6721
Clean: 8369


### Pick one clean and one hallucinated example

In [5]:
def parse_labels(x):
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, dict):
        return [x]
    if isinstance(x, str):
        s = x.strip()
        if not s or s.lower() == "nan":
            return []
        for parser in (json.loads, ast.literal_eval):
            try:
                val = parser(s)
                if isinstance(val, list):
                    return val
                if isinstance(val, dict):
                    return [val]
            except Exception:
                pass
    return []

# Pick a clean example (id=0)
clean_row = df[~df["has_hallucination"]].iloc[0]
# Pick a hallucinated example (id=2, mistral with evident_conflict + baseless_info)
hall_row = df[df["has_hallucination"]].iloc[0]

print("=" * 60)
print("CLEAN EXAMPLE")
print("=" * 60)
print(f"id: {clean_row['id']}, model: {clean_row['model']}")
print(f"evident_conflict: {clean_row['evident_conflict']}, baseless_info: {clean_row['baseless_info']}")
print(f"\nQuery: {clean_row['query'][:100]}...")
print(f"\nAnswer: {clean_row['answer'][:200]}...")

print("\n" + "=" * 60)
print("HALLUCINATED EXAMPLE")
print("=" * 60)
print(f"id: {hall_row['id']}, model: {hall_row['model']}")
print(f"evident_conflict: {hall_row['evident_conflict']}, baseless_info: {hall_row['baseless_info']}")
print(f"\nQuery: {hall_row['query'][:100]}...")
print(f"\nAnswer: {hall_row['answer'][:200]}...")

hall_labels = parse_labels(hall_row["hallucination_labels"])
print(f"\nHallucination spans:")
for lab in hall_labels:
    if isinstance(lab, dict):
        print(f"  - [{lab.get('label_type')}] {lab.get('text', '')[:80]}")

CLEAN EXAMPLE
id: 0, model: gpt-4-0613
evident_conflict: 0, baseless_info: 0

Query: Summarize the following news within 116 words:...

Answer: The Anne Frank House has revealed that Anne Frank and her older sister, Margot, likely died at least a month earlier than previously believed. The sisters, who were imprisoned in Nazi concentration ca...

HALLUCINATED EXAMPLE
id: 2, model: mistral-7B-instruct
evident_conflict: 1, baseless_info: 1

Query: Summarize the following news within 116 words:...

Answer: New research conducted by the Anne Frank House has revealed that Anne Frank and her sister Margot likely died in the Bergen-Belsen concentration camp at least a month earlier than previously believed....

Hallucination spans:
  - [Evident Conflict] February 7, 2022.
  - [Evident Baseless Info] has prompted the Anne Frank House to issue a corrected statement regarding the d
  - [Evident Conflict] believed to have died before February 7


## 3. Run ContextGroundednessScorer

In [6]:
from uqlm import ContextGroundednessScorer

scorer = ContextGroundednessScorer(
    nli_llm=llm,
    entailment_style="nli_classification",
    aggregation="mean",
)
print("Scorer created successfully")

Scorer created successfully


In [7]:
queries = [clean_row["query"], hall_row["query"]]
contexts = [clean_row["context"], hall_row["context"]]
answers = [clean_row["answer"], hall_row["answer"]]

result = await scorer.score(
    queries=queries,
    contexts=contexts,
    answers=answers,
    return_raw_judge_responses=True,
)
print("Scoring complete!")

Scoring complete!


## 4. Inspect Results

In [16]:
print(f"Query:\n{queries[0]}\n\nContext:\n{contexts[0][:500]}")

example_names = ["CLEAN", "HALLUCINATED"]
gt_labels = [
    {"evident_conflict": clean_row["evident_conflict"], "baseless_info": clean_row["baseless_info"]},
    {"evident_conflict": hall_row["evident_conflict"], "baseless_info": hall_row["baseless_info"]},
]

for i, name in enumerate(example_names):
    print("=" * 70)
    print(f"{name} EXAMPLE (response_score={result.response_scores[i]:.3f})")
    print(f"Ground truth: {gt_labels[i]}")
    print("=" * 70)

    claims = result.claim_sets[i]
    labels = result.claim_labels[i]
    scores = result.claim_scores[i]

    for j, (claim, label, score) in enumerate(zip(claims, labels, scores)):
        icon = "✅" if label == "supported" else "❌" if label == "contradiction" else "❓" if label == "baseless" else "⚠️"
        print(f"  {icon} [{label:>13s}] (score={score:.1f}) {claim[:100]}")

    if name == "HALLUCINATED":
        hall_labels = parse_labels(hall_row["hallucination_labels"])
        print(f"\nHallucination spans:")
        for lab in hall_labels:
            if isinstance(lab, dict):
                print(f"  - [{lab.get('label_type')}] {lab.get('text', '')[:80]}")
    print()

Query:
Summarize the following news within 116 words:

Context:
Seventy years ago, Anne Frank died of typhus in a Nazi concentration camp at the age of 15. Just two weeks after her supposed death on March 31, 1945, the Bergen-Belsen concentration camp where she had been imprisoned was liberated -- timing that showed how close the Jewish diarist had been to surviving the Holocaust. But new research released by the Anne Frank House shows that Anne and her older sister, Margot Frank, died at least a month earlier than previously thought. Researchers re-examined
CLEAN EXAMPLE (response_score=0.833)
Ground truth: {'evident_conflict': 0, 'baseless_info': 0}
  ✅ [    supported] (score=1.0) The Anne Frank House revealed that Anne Frank and Margot likely died at least a month earlier than p
  ✅ [    supported] (score=1.0) Anne Frank and Margot were imprisoned in Nazi concentration camps during the Holocaust.
  ✅ [    supported] (score=1.0) Anne Frank and Margot were thought to have died in Marc

## 5. Summary

In [9]:
print("Response-level scores:")
for i, name in enumerate(example_names):
    n_claims = len(result.claim_sets[i])
    n_supported = sum(1 for l in result.claim_labels[i] if l == "supported")
    n_contradiction = sum(1 for l in result.claim_labels[i] if l == "contradiction")
    n_baseless = sum(1 for l in result.claim_labels[i] if l == "baseless")
    n_unknown = sum(1 for l in result.claim_labels[i] if l == "unknown")

    print(f"\n{name}:")
    print(f"  Response score: {result.response_scores[i]:.3f}")
    print(f"  Total claims: {n_claims}")
    print(f"  Supported: {n_supported}, Contradiction: {n_contradiction}, Baseless: {n_baseless}, Unknown: {n_unknown}")
    print(f"  GT: evident_conflict={gt_labels[i]['evident_conflict']}, baseless_info={gt_labels[i]['baseless_info']}")

Response-level scores:

CLEAN:
  Response score: 0.833
  Total claims: 9
  Supported: 7, Contradiction: 1, Baseless: 1, Unknown: 0
  GT: evident_conflict=0, baseless_info=0

HALLUCINATED:
  Response score: 0.900
  Total claims: 10
  Supported: 8, Contradiction: 0, Baseless: 2, Unknown: 0
  GT: evident_conflict=1, baseless_info=1


In [10]:
# Show raw judge responses for debugging
if result.metadata.get("raw_judge_responses"):
    for i, name in enumerate(example_names):
        print(f"\n{'=' * 40} {name} RAW RESPONSES {'=' * 40}")
        for j, (claim, raw) in enumerate(zip(result.claim_sets[i], result.metadata["raw_judge_responses"][i])):
            print(f"  Claim: {claim[:80]}")
            print(f"  Judge: {raw[:100]}")
            print()


======================================== CLEAN RAW RESPONSES ========================================
  Claim: The Anne Frank House revealed that Anne Frank and Margot likely died at least a 
  Judge: entailment

  Claim: Anne Frank and Margot were imprisoned in Nazi concentration camps during the Hol
  Judge: entailment

  Claim: Anne Frank and Margot were thought to have died in March 1945.
  Judge: entailment

  Claim: Anne Frank and Margot died two weeks before the Bergen-Belsen camp was liberated
  Judge: contradiction

  Claim: New research suggests Anne Frank and Margot did not survive until March 1945.
  Judge: entailment

  Claim: The research examined archives from the Red Cross, the International Tracing Ser
  Judge: neutral

  Claim: The exact dates of Anne Frank and Margot's deaths remain unclear.
  Judge: entailment

  Claim: Anne Frank and Margot had symptoms of typhus before February 7, 1945.
  Judge: entailment

  Claim: Anne Frank and Margot succumbed to typhus.
  Ju